# Inspect Consolidated Soil Moisture Datasets

Load and visualize the four soil moisture source datasets for January 1980.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

RUN_DIR = Path.home() / "data" / "test1" / "2026-03-12_test1_v0.1.0"


## Load all four datasets

In [ ]:
datasets = {
    "MERRA-2 (GWETTOP)": {
        "path": RUN_DIR / "data" / "raw" / "merra2" / "merra2_consolidated.nc",
        "var": "GWETTOP",
        "units": "fraction (0-1)",
    },
    "NLDAS-2 MOSAIC (SoilM 0-10cm)": {
        "path": RUN_DIR / "data" / "raw" / "nldas_mosaic" / "nldas_mosaic_consolidated.nc",
        "var": "SoilM_0_10cm",
        "units": "kg/m²",
    },
    "NLDAS-2 NOAH (SoilM 0-10cm)": {
        "path": RUN_DIR / "data" / "raw" / "nldas_noah" / "nldas_noah_consolidated.nc",
        "var": "SoilM_0_10cm",
        "units": "kg/m²",
    },
    "NCEP/NCAR (soilw 0-10cm)": {
        "path": RUN_DIR / "data" / "raw" / "ncep_ncar" / "ncep_ncar_consolidated.nc",
        "var": "soilw_0_10cm",
        "units": "kg/m²",
    },
}

# Open each consolidated file and report basic info
opened = {}
for label, info in datasets.items():
    nc_path = info["path"]
    if not nc_path.exists():
        print(f"SKIP {label}: {nc_path} not found (run consolidation first)")
        continue
    ds = xr.open_dataset(nc_path)
    opened[label] = (ds, info)
    print(f"{label}: {list(ds.data_vars)} | time: {ds.time.values[0]} .. {ds.time.values[-1]} | shape: {dict(ds.sizes)}")


## Plot January 1980

In [ ]:
TARGET_TIME = "1980-01-15"

n = len(opened)
if n == 0:
    print("No datasets available yet. Run the fetch commands first.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()

    for idx, (label, (ds, info)) in enumerate(opened.items()):
        ax = axes[idx]
        var = info["var"]
        da = ds[var].sel(time=TARGET_TIME, method="nearest")
        actual_time = str(da.time.values)[:10]

        da.plot(ax=ax, cmap="YlGnBu", robust=True)
        ax.set_title(f"{label}\n{actual_time} | {info['units']}", fontsize=11)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    # Hide unused subplots
    for idx in range(len(opened), 4):
        axes[idx].set_visible(False)

    fig.suptitle(f"Soil Moisture — nearest to {TARGET_TIME}", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


## Clean up

In [ ]:
for label, (ds, _) in opened.items():
    ds.close()
